# Calculate effective coverage by quintile and scenario

Also by age, sex, and pregnancy status (though coverage will not vary by pregnancy status due to a lack of data).

This is similar to what the pregnancy simulation does at the individual level, but using groups instead.
It can be shared between multiplication models that do not incorporate individual heterogeneity.

In [ ]:
import pandas as pd

In [ ]:
location = "nigeria"
fortificant = "iron"

In [ ]:
results_dir = f'../results/{fortificant}'

In [ ]:
full_coverage_probability = pd.read_csv(f'{results_dir}/baseline_fortification/full_coverage/{location}.csv')
full_coverage_probability = full_coverage_probability.set_index([c for c in full_coverage_probability.columns if c != 'value']).value
full_coverage_probability

In [ ]:
any_coverage_probability = pd.read_csv(f'{results_dir}/baseline_fortification/any_coverage/{location}.csv')
any_coverage_probability = any_coverage_probability.set_index([c for c in any_coverage_probability.columns if c != 'value']).value
any_coverage_probability

In [ ]:
partial_coverage_mean = pd.read_csv(f'{results_dir}/baseline_fortification/partial_coverage_amount/mean/{location}.csv')
partial_coverage_mean = partial_coverage_mean.set_index([c for c in partial_coverage_mean.columns if c != 'value']).value
partial_coverage_mean

In [ ]:
current_coverage = full_coverage_probability + (any_coverage_probability - full_coverage_probability) * partial_coverage_mean
current_coverage

In [ ]:
scenarios = {
    "india": ["intervention"],
    "nigeria": ["intervention"],
    "ethiopia": ["intervention_25_nrv", "intervention_100_nrv"],
}[location]

In [ ]:
fortifiability = pd.read_csv(f'{results_dir}/../vehicle_consumption/fortifiability/{location}.csv')
fortifiability = fortifiability.set_index([c for c in fortifiability.columns if c != 'value']).value
fortifiability

In [ ]:
import pathlib

for scenario in scenarios:
    intervention_coverage = pd.read_csv(f'{results_dir}/{scenario}/intervention_fortification/any_coverage/{location}.csv')
    intervention_coverage = intervention_coverage.set_index([c for c in intervention_coverage.columns if c != 'value']).value
    target_coverage = intervention_coverage * fortifiability
    display(target_coverage)
    assert (target_coverage > current_coverage.reindex_like(target_coverage)).all()
    # Not all coverage is effective -- this is as a proportion of coverage!
    effective_coverage = pd.read_csv(f'{results_dir}/{scenario}/intervention_fortification/effective_coverage/{location}.csv')
    effective_coverage = effective_coverage.set_index([c for c in effective_coverage.columns if c != 'value']).value
    effective_intervention_coverage = target_coverage * effective_coverage
    path = f'{results_dir}/{scenario}/effective_intervention_coverage/{location}.csv'
    pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
    effective_intervention_coverage.reset_index().to_csv(path, index=False)


In [ ]:
effective_baseline_coverage = current_coverage * effective_coverage
effective_baseline_coverage

In [ ]:
path = f'{results_dir}/effective_baseline_coverage/{location}.csv'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
effective_baseline_coverage.reset_index().to_csv(path, index=False)